In [2]:
import os
import certifi
import requests

from pydantic_settings import BaseSettings
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from langchainhub import Client

C:\Users\Admin\AppData\Local\Temp\ipykernel_15968\1743884499.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [4]:
from langchain.agents import create_agent

class Settings(BaseSettings):
    GEMINI_API_KEY: str | None = None
    TAVILY_API_KEY: str | None = None
    model_config = {"env_file": ".env", "extra": "ignore"}
    
    
settings = Settings()

In [5]:
search_tool = TavilySearchResults(max_results=3, api_key=settings.TAVILY_API_KEY)

C:\Users\Admin\AppData\Local\Temp\ipykernel_15968\813368507.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=3, api_key=settings.TAVILY_API_KEY)


In [6]:
search_tool.invoke("Thủ đô của Việt Nam là tỉnh nào?")

[{'title': 'Thủ đô Việt Nam – Wikipedia tiếng Việt',
  'url': 'https://vi.wikipedia.org/wiki/Th%E1%BB%A7_%C4%91%C3%B4_Vi%E1%BB%87t_Nam',
  'content': 'Hiện nay Việt Nam có 5 tỉnh, thành được gọi là các "vùng kinh đô" gồm: Hà Nội, Phú Thọ, Ninh Bình, Thanh Hóa và Thừa Thiên – Huế. 5 vùng kinh đô này được ngành văn hóa cho phép tổ chức và tham gia nhiều sự kiện lớn như: cuộc thi người đẹp các vùng kinh đô, hiệp hội văn học nghệ thuật các vùng kinh đô, triển lãm ảnh ngũ đại cố đô của Việt Nam, Hành trình di sản thế giới... Năm du lịch Quốc gia 2015 diễn ra ở Thanh Hóa và các tỉnh, thành phố có Kinh đô cổ và di sản văn hoá thế giới có chuyên đề "Hành trình về Kinh đô cổ Việt Nam".\n\n## Xem thêm\n\n## Tham khảo\n\nWikimedia Foundation\nPowered by MediaWiki [...] ## Cố đô của Việt Nam\n\nCố đô là cách gọi tôn vinh những nơi từng là thủ đô chính thống trong lịch sử Việt Nam. Hiện ở Việt Nam có các nơi sau được gọi là cố đô gồm: đất tổ Phong Châu "Phong Châu (kinh đô)"), cố đô Hoa Lư "Hoa Lư 

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", 
    api_key=settings.GEMINI_API_KEY,
    temperature=0.0)

In [9]:
response =llm.invoke("Hôm nay là ngày mấy?")
response

AIMessage(content=[{'type': 'text', 'text': 'Hôm nay là **thứ Ba, ngày 22 tháng 10 năm 2024**.', 'extras': {'signature': 'EjQKMgERTTIPNbln9x4/ookjxHsFc/dFvojP27GrYG8oZ8QJsH2iGWW+QBrsvJKyDOmGCZqw'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fc6a3-aec4-7cd0-96bd-8c1df68e8a4c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 24, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}})

In [ ]:
#=====================
# TẠO PROMPT TEMPLATE
#=====================

from langchain_core.prompts import PromptTemplate



# Định nghĩa template chuỗi (nội dung chi tiết có tại hwchase17/react)
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

# Khởi tạo PromptTemplate từ chuỗi
prompt = PromptTemplate.from_template(react_template)
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:
#=====================
# TẠO TOOLS
#=====================

tools = [search_tool]

In [ ]:
#=====================
# TAO AGENT
#=====================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful AI assistant. Answer user questions using the provided tools when you need to search the web for information."
)